In [3]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\dmcol\AppData\Local\Programs\Python\Python311\python.exe
3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]


In [4]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [5]:
import cv2
from ultralytics import YOLO
from collections import defaultdict
import easyocr

ImportError: cannot import name 'NP_SUPPORTED_MODULES' from 'torch._dynamo.utils' (c:\Users\dmcol\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\utils.py)

In [ ]:
# ── Load model ───────────────────────────────────────────────
model = YOLO("E:/Github/Number plate detection/Source_Files_and_final_weights/Final_weights/best.pt")
reader = easyocr.Reader(['en'], gpu=False) 

Using CPU. Note: This module is much faster with a GPU.


In [6]:
for i in range(5):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        print(f"Camera found at index {i}")
        cap.release()
    else:
        print(f"No camera at index {i}")

No camera at index 0
No camera at index 1
Camera found at index 2
Camera found at index 3
No camera at index 4


In [7]:
# ── Open camera ──────────────────────────────────────────────
cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

frame_count = 0
last_boxes  = []
last_texts  = {}   # store last OCR text per box index

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # ── Run YOLO every 2nd frame ─────────────────────────────
    if frame_count % 2 == 0:
        results    = model.predict(
            source  = frame,
            imgsz   = 320,
            conf    = 0.4,
            verbose = False
        )
        last_boxes = results[0].boxes

        # ── Run OCR every 6th frame (OCR is slow) ────────────
        if frame_count % 6 == 0 and last_boxes is not None:
            last_texts = {}
            for i, box in enumerate(last_boxes):
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # add padding around plate for better OCR
                pad = 4
                x1p = max(0, x1 - pad)
                y1p = max(0, y1 - pad)
                x2p = min(frame.shape[1], x2 + pad)
                y2p = min(frame.shape[0], y2 + pad)

                plate_crop = frame[y1p:y2p, x1p:x2p]

                if plate_crop.size == 0:
                    continue

                # preprocess crop for better OCR accuracy
                gray    = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
                resized = cv2.resize(gray, None, fx=2, fy=2)  # upscale 2x

                ocr_result = reader.readtext(resized)

                if ocr_result:
                    # take the highest confidence OCR result
                    best = max(ocr_result, key=lambda x: x[2])
                    text = best[1].upper().strip()
                    conf = best[2]
                    if conf > 0.3:                 # only keep confident reads
                        last_texts[i] = text

    # ── Draw boxes and OCR text on every frame ───────────────
    if last_boxes is not None:
        for i, box in enumerate(last_boxes):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf_score = float(box.conf[0])

            # green box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 80), 2)

            # detection confidence label
            det_label = f"Plate {conf_score:.2f}"
            (lw, lh), _ = cv2.getTextSize(det_label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(frame, (x1, y1 - lh - 8), (x1 + lw + 4, y1), (0, 255, 80), -1)
            cv2.putText(frame, det_label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

            # OCR text below the box
            if i in last_texts:
                ocr_label = last_texts[i]
                (tw, th), _ = cv2.getTextSize(ocr_label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y2), (x1 + tw + 4, y2 + th + 8), (255, 140, 0), -1)
                cv2.putText(frame, ocr_label, (x1 + 2, y2 + th + 2),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # ── Info overlay ─────────────────────────────────────────
    cv2.putText(frame, f"Plates: {len(last_boxes) if last_boxes else 0}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 80), 2)

    cv2.imshow("License Plate Detection + OCR", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


In [8]:
import cv2
print(cv2.__version__)

4.13.0


In [2]:
import cv2
info = cv2.getBuildInformation()
print("GUI support:" , "YES" if "GTK" in info or "WIN32 UI" in info else "NO - headless version")

GUI support: NO - headless version
